# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx spacy datasets langchain-community llama-index

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: E:\aitc\w4\K4-DAY19-2A202601242-HoangTrongDai\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "data/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "data/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

if os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 10 * 1024 * 1024:
    print(f"File đã tồn tại, bỏ qua streaming: {OUTPUT_CSV} ({os.path.getsize(OUTPUT_CSV)/(1024*1024):.2f} MB)")
    DATA_PATH = OUTPUT_CSV
else:
    print("Đang kết nối luồng dữ liệu (streaming)...")

    try:
        dataset = load_dataset(
            DATASET_NAME,
            split="train",
            streaming=True,
            token=HF_TOKEN,
        )
        iterator = iter(dataset)

        first_row = next(iterator)
        headers = list(first_row.keys())

        print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

        rows_written = 0
        total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
        unit_progress = "MB" if PRIORITIZE_MB else "row"

        with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
            writer.writeheader()
            writer.writerow(first_row)
            rows_written += 1

            # Flush để kích thước file phản ánh dữ liệu vừa ghi.
            f.flush()
            file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

            with tqdm(
                total=total_progress,
                desc=f"Đang tải ({unit_progress})",
                unit=unit_progress,
            ) as pbar:
                if PRIORITIZE_MB:
                    pbar.n = min(file_size_mb, LIMIT_MB)
                    pbar.refresh()
                else:
                    pbar.update(1)

                for row in iterator:
                    writer.writerow(row)
                    rows_written += 1

                    # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                    # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                    should_check_size = (
                        PRIORITIZE_MB
                        and (
                            rows_written % 100 == 0
                            or file_size_mb >= LIMIT_MB * 0.95
                        )
                    )

                    if should_check_size:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                        pbar.refresh()
                    elif not PRIORITIZE_MB:
                        pbar.update(1)

                    # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                    if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                        print(
                            f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                            f"{file_size_mb:.2f} MB "
                            f"(Tổng: {rows_written:,} dòng)"
                        )
                        break

                    # Hard-stop theo số dòng trong mọi chế độ.
                    if rows_written >= LIMIT_ROWS:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        print(
                            f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                            f"{rows_written:,} dòng "
                            f"(Dung lượng: {file_size_mb:.2f} MB)"
                        )
                        break

            f.flush()

        final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
        print(
            f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
            f"   Rows: {rows_written:,}\n"
            f"   Size: {final_size_mb:.2f} MB"
        )

        # Đồng bộ đường dẫn cho cell loader tiếp theo.
        DATA_PATH = OUTPUT_CSV

    except StopIteration:
        raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
    except Exception as e:
        print(f"\n❌ Có lỗi xảy ra: {e}")
        print(
            "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
            "(3) kết nối mạng của Colab."
        )
        raise

File đã tồn tại, bỏ qua streaming: data/hackernoon_subset.csv (300.00 MB)


In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.


✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Exact dedup: 245,324 -> 212,212


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,d38fc817b7dc3eeeb535::c0000,d38fc817b7dc3eeeb535,Information Services Corporation Non-GAAP EPS of C$0.51 revenue of C$53.3M,2023-08-03,To ensure this doesn’t happen in the future please enable Javascript and cookies in your browser. Is this happening ...
1,830dbc6ea3082bb64be0::c0000,830dbc6ea3082bb64be0,How GSA’s Technology Transformation Services is Harnessing Change in Tech Modernization,2023-05-24,One component of GSA in particular Technology Transformation Services carries much of this mission by using modern m...
2,8cbfe069b03305566d73::c0000,8cbfe069b03305566d73,Information Technology,2023-05-18,At the most recent Berkshire Hathaway Inc. (NYSE: BRK-B) investors conference in early May Warren Buffett offered so...
3,331537d8f978a369b442::c0000,331537d8f978a369b442,Ryan Specialty Signs Definitive Agreement To Acquire Socius Insurance,2023-05-23,Ryan Specialty (NYSE:RYAN) a leading international specialty insurance firm is pleased to announce that it has signe...
4,55a09cbc43c41ffb2dd9::c0000,55a09cbc43c41ffb2dd9,Transact Campus Partnership Lands Talkiatry Services on Campus Transact Apps,2023-09-05,The partnership will “provide students with access to quality psychiatric services and offers an accessible and affo...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

COREF_CHECKPOINT = "data/_ckpt_coref.csv"
extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
if os.path.exists(COREF_CHECKPOINT):
    print(f"Checkpoint found, bo qua goi LLM: {COREF_CHECKPOINT}")
    coref_df = pd.read_csv(COREF_CHECKPOINT)
    coref_df["unresolved_mentions"] = coref_df["unresolved_mentions"].apply(
        lambda x: json.loads(x) if isinstance(x, str) and x.strip().startswith("[") else []
    )
else:
    coref_df = run_coref(extraction_source)
    _save = coref_df.copy()
    _save["unresolved_mentions"] = _save["unresolved_mentions"].apply(json.dumps)
    _save.to_csv(COREF_CHECKPOINT, index=False)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Checkpoint found, bo qua goi LLM: data/_ckpt_coref.csv


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    return pd.DataFrame(triples), pd.DataFrame(errors)

TRIPLES_CHECKPOINT = "data/_ckpt_triples.csv"
if os.path.exists(TRIPLES_CHECKPOINT):
    print(f"Checkpoint found, bo qua goi LLM: {TRIPLES_CHECKPOINT}")
    raw_triples_df = pd.read_csv(TRIPLES_CHECKPOINT)
    extraction_errors_df = pd.DataFrame()
else:
    raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
    if raw_triples_df.empty:
        raise RuntimeError(
            f"Extraction tra ve 0 triples ({len(extraction_errors_df)} loi). "
            f"Sample loi: {extraction_errors_df.head(3).to_dict('records') if not extraction_errors_df.empty else 'N/A'}"
        )
    raw_triples_df.to_csv(TRIPLES_CHECKPOINT, index=False)
display(raw_triples_df.head())

Checkpoint found, bo qua goi LLM: data/_ckpt_triples.csv


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Ryan Specialty,Company,ACQUIRED,Socius Insurance Services,Company,331537d8f978a369b442::c0000,2023-05-23,Ryan Specialty (NYSE:RYAN) ... has signed a definitive agreement to acquire Socius Insurance Services,1.0
1,Talkiatry,Company,PARTNERED_WITH,Transact,Company,55a09cbc43c41ffb2dd9::c0000,2023-09-05,Talkiatry said... Transact said it will provide partner institutions...,1.0
2,Synchron,Company,USES,brain-computer interface,Technology,a7a8a5531a30bc5c7fbb::c0000,2023-02-18,Synchron is part of an emerging crop of companies testing technology in the brain-computer interface industry.,0.9
3,Sensormatic Solutions,Company,PARTNERED_WITH,Zliide,Company,adb56add69f473fc4105::c0000,2023-01-16,Sensormatic Solutions ... announced its collaboration with Zliide a technology company...,1.0
4,ServiceNow Inc.,Company,DEVELOPED,Utah release,Technology,3a1191cd49142548e77e::c0000,2023-03-22,ServiceNow Inc. is rolling out a new version of its Now platform ... known as the Utah release,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.40, top_k=6):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

AUDIT_CHECKPOINT = "data/_ckpt_entity_audit.csv"
CANON_CHECKPOINT = "data/_ckpt_canonical_triples.csv"
if os.path.exists(AUDIT_CHECKPOINT) and os.path.exists(CANON_CHECKPOINT):
    print("Checkpoint found, bo qua tinh lai.")
    entity_resolution_audit_df = pd.read_csv(AUDIT_CHECKPOINT)
    triples_df = pd.read_csv(CANON_CHECKPOINT)
else:
    entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
    triples_df = canonicalize_triples(raw_triples_df, entity_map)
    entity_resolution_audit_df.to_csv(AUDIT_CHECKPOINT, index=False)
    triples_df.to_csv(CANON_CHECKPOINT, index=False)
display(entity_resolution_audit_df.head(20))

Checkpoint found, bo qua tinh lai.


,type,left,right,similarity,decision
0,Technology,Mitsubishi Tensei 1K,Mitsubishi Tensei 1K Black 65 TX,0.843918,MERGE_VECTOR
1,Technology,AI,Artificial Intelligence,0.791246,REJECT_GUARD
2,Person,Pyotr Ufimtsev,Ufimtsev,0.770834,MERGE_VECTOR
3,Technology,Annealing quantum computing,Gate model quantum computing,0.625415,MERGE_VECTOR
4,Company,Nexsyst 360,Nexxiot,0.582073,REJECT_GUARD
5,Person,Manolo Saiz,Roman Saini,0.553505,REJECT_GUARD
6,Technology,AI,AI-powered avatars,0.545086,REJECT_GUARD
7,Company,NIO Inc.,ServiceNow Inc.,0.536527,REJECT_GUARD
8,Person,Isha Ambani,Rajiv Mehrishi,0.536356,REJECT_GUARD
9,Person,Gaurav Munjal,Sriram Krishnan,0.521170,REJECT_GUARD


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 103, 'edges': 62, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,e6c3db5c28c9a15bfafd2e04,Apple,Company,5
1,7b219357277193c25d37d788,Railergy,Company,5
2,8693b055457df450df9cfcdf,Unacademy,Company,3
3,f87c100acdc217a40f2716db,Max Homa,Person,3
4,1a8e38440c5cb62e7368d374,Richard Gonzalez,Person,2
5,a606269da51957359d0c9e43,stealth technology,Technology,2
6,8a9f24fbb48a8c90768ed465,Robot Attorney,Technology,2
7,909fcd9c188c8c2429afa468,Google Cloud,Company,2
8,9d5ca34779b3c2404977a25e,D-Wave Quantum Inc.,Company,2
9,39430d6a0c7aa28d8fe56bf3,Meeno,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [14]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [15]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [16]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = "data/golden_dataset.csv"

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"General knowledge check - not expected to be found in this random HackerNoon subset (tests hallucination control)."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startup was founded by Renate Nyborg, and which investor led its seed funding round?",
        "reference_answer":"Renate Nyborg founded the startup Meeno. Sequoia led a $3.9 million seed funding round for Meeno.",
        "reference_evidence":"chunk=212476770ac78703d3b7::c0000 (2023-09-27): 'Renate Nyborg's startup Meeno' + 'Meeno has raised a $3.9 million seed round led by Sequoia.'"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare two startup funding events reported in September 2023: Meeno and Aaritya Technologies. Who invested, and how do the deals differ?",
        "reference_answer":"In September 2023 two early-stage startups raised seed funding: (1) Meeno, founded by Renate Nyborg, raised a $3.9 million seed round led by Sequoia (2023-09-27). (2) Aaritya Technologies, the new venture of ex-Swiggy CTO Dale Vaz, received seed funding from venture funds Accel and Elevation Capital (2023-09-08). Both are VC-led seed rounds in the same month, but from different investors, founders, and verticals.",
        "reference_evidence":"chunk=212476770ac78703d3b7::c0000 (2023-09-27) and chunk=6ef23bfa65272a9d88a4::c0000 (2023-09-08)."
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Identify a company whose product is itself powered by another named technology, and describe the full chain of relations and dates.",
        "reference_answer":"DoNotPay uses its 'Robot Attorney' product (2023-01-24), and Robot Attorney itself uses/is powered by Artificial Intelligence (same date, same article). Chain: DoNotPay -USES-> Robot Attorney -USES-> Artificial Intelligence.",
        "reference_evidence":"chunk=db1d16a11c2b46df6faf::c0000 (2023-01-24): 'The technology is available to consumers via the smartphone application called DoNotPay' + 'robot attorney powered by artificial intelligence'."
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Based on multiple articles from 2023, how did Apple's technology activities evolve over the year?",
        "reference_answer":"Apple's 2023 coverage shows a shift from creative software to silicon/hardware partnerships: in May 2023 Apple brought Final Cut Pro and Logic Pro editing software to iPad; by August 2023 Apple had developed the M3 processor line and the A17 Bionic chip (iPhone 15); by September 2023 Apple signed a chip-technology partnership deal with Arm.",
        "reference_evidence":"chunk=b18d823694ce9d481434::c0000 (2023-05-24), chunk=e5b7afc0b65c1244d423::c0000 (2023-08-21), chunk=ba79631be3da045c5f17::c0000 (2023-09-07)."
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,question,reference_answer,reference_evidence
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,General knowledge check - not expected to be found in this random HackerNoon subset (tests hallucination control).
1,G02,multi-hop,"Which startup was founded by Renate Nyborg, and which investor led its seed funding round?",Renate Nyborg founded the startup Meeno. Sequoia led a $3.9 million seed funding round for Meeno.,chunk=212476770ac78703d3b7::c0000 (2023-09-27): 'Renate Nyborg's startup Meeno' + 'Meeno has raised a $3.9 million s...
2,G03,cross-doc,"Compare two startup funding events reported in September 2023: Meeno and Aaritya Technologies. Who invested, and how...","In September 2023 two early-stage startups raised seed funding: (1) Meeno, founded by Renate Nyborg, raised a $3.9 m...",chunk=212476770ac78703d3b7::c0000 (2023-09-27) and chunk=6ef23bfa65272a9d88a4::c0000 (2023-09-08).
3,G04,multi-hop,"Identify a company whose product is itself powered by another named technology, and describe the full chain of relat...","DoNotPay uses its 'Robot Attorney' product (2023-01-24), and Robot Attorney itself uses/is powered by Artificial Int...",chunk=db1d16a11c2b46df6faf::c0000 (2023-01-24): 'The technology is available to consumers via the smartphone applica...
4,G05,cross-doc,"Based on multiple articles from 2023, how did Apple's technology activities evolve over the year?",Apple's 2023 coverage shows a shift from creative software to silicon/hardware partnerships: in May 2023 Apple broug...,"chunk=b18d823694ce9d481434::c0000 (2023-05-24), chunk=e5b7afc0b65c1244d423::c0000 (2023-08-21), chunk=ba79631be3da04..."


In [17]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [18]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = "outputs/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df):
    rows = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=False)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G01,factoid,Who was the CEO of Hugging Face in 2023?,Clément Delangue,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Who was the CEO of Hugging Fa...,<think>\nHere's a thinking process:\n\n1. **Analyze User Input:**\n - **Question:** Who was the CEO of Hugging Fa...,1,1,1,1,1,1,2.500393,2.098818,1787,1473,The candidate response correctly identifies that the provided context does not contain any information about the CEO...,The candidate response correctly identifies that the provided context does not contain any information about the CEO...,0
1,G02,multi-hop,"Which startup was founded by Renate Nyborg, and which investor led its seed funding round?",Renate Nyborg founded the startup Meeno. Sequoia led a $3.9 million seed funding round for Meeno.,"<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Question: ""Which startup was founded by ...",<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Which startup was founded by Renate Nybo...,5,5,5,5,5,5,2.552515,2.388132,1820,1787,"The candidate provides a complete and accurate answer to the question, identifying both the startup founded by Renat...","The candidate provides a complete and accurate answer to the question, identifying both the startup founded by Renat...",0
2,G03,cross-doc,"Compare two startup funding events reported in September 2023: Meeno and Aaritya Technologies. Who invested, and how...","In September 2023 two early-stage startups raised seed funding: (1) Meeno, founded by Renate Nyborg, raised a $3.9 m...",<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Compare two startup funding events in Se...,<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Compare two startup funding events in Se...,1,1,1,1,1,1,3.438476,8.419505,2293,2207,"The candidate response accurately reflects the information provided in the context regarding Meeno's funding event, ...",The candidate answer provides only the information available about Meeno and explicitly states the lack of informati...,0
3,G04,multi-hop,"Identify a company whose product is itself powered by another named technology, and describe the full chain of relat...","DoNotPay uses its 'Robot Attorney' product (2023-01-24), and Robot Attorney itself uses/is powered by Artificial Int...",<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Identify a company whose product is itse...,<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - Identify a company whose product is itse...,1,1,1,1,1,1,3.991174,22.723128,2488,2479,The candidate response correctly identifies the lack of evidence in the supplied context to answer the question. It ...,The candidate response correctly identifies that the provided context does not contain the necessary information to ...,0
4,G05,cross-doc,"Based on multiple articles from 2023, how did Apple's technology activities evolve over the year?",Apple's 2023 coverage shows a shift from creative software to silicon/hardware partnerships: in May 2023 Apple broug...,"<think>\nHere's a thinking process:\n\n1. **Analyze User Question:** The user asks: ""Based on multiple articles fro...","<think>\nHere's a thinking process:\n\n1. **Analyze User Question:** The user asks: ""Based on multiple articles fro...",2,5,5,5,3,5,14.535461,4.188892,2643,2852,"The candidate provides isolated data points regarding Apple's technology activities in 2023, but it lacks a comprehe...","The candidate provides a detailed chronological account of Apple's technology activities in 2023, accurately reflect...",0


In [19]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("outputs/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("outputs/graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.500,3.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
1,cross-doc,Faithfulness,3.000,3.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,2.000,3.000,GraphRAG cải thiện rõ; kiểm tra rationale và provenance.
3,cross-doc,Latency (s),8.987,6.304,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,2468.000,2529.500,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),2.500,2.099,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,1787.000,1473.000,GraphRAG không đắt hơn trong sample này.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [20]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': '7b219357277193c25d37d788', 'name': 'Railergy', 'degree': 5} fetched= 5


,type,left,right,similarity,decision
0,Technology,Mitsubishi Tensei 1K,Mitsubishi Tensei 1K Black 65 TX,0.843918,MERGE_VECTOR
1,Technology,AI,Artificial Intelligence,0.791246,REJECT_GUARD
2,Person,Pyotr Ufimtsev,Ufimtsev,0.770834,MERGE_VECTOR
3,Technology,Annealing quantum computing,Gate model quantum computing,0.625415,MERGE_VECTOR
4,Company,Nexsyst 360,Nexxiot,0.582073,REJECT_GUARD
5,Person,Manolo Saiz,Roman Saini,0.553505,REJECT_GUARD
6,Technology,AI,AI-powered avatars,0.545086,REJECT_GUARD
7,Company,NIO Inc.,ServiceNow Inc.,0.536527,REJECT_GUARD
8,Person,Isha Ambani,Rajiv Mehrishi,0.536356,REJECT_GUARD
9,Person,Gaurav Munjal,Sriram Krishnan,0.521170,REJECT_GUARD


High-similarity rejected pairs:


,type,left,right,similarity,decision
1,Technology,AI,Artificial Intelligence,0.791246,REJECT_GUARD
4,Company,Nexsyst 360,Nexxiot,0.582073,REJECT_GUARD
5,Person,Manolo Saiz,Roman Saini,0.553505,REJECT_GUARD
6,Technology,AI,AI-powered avatars,0.545086,REJECT_GUARD
7,Company,NIO Inc.,ServiceNow Inc.,0.536527,REJECT_GUARD
8,Person,Isha Ambani,Rajiv Mehrishi,0.536356,REJECT_GUARD
9,Person,Gaurav Munjal,Sriram Krishnan,0.521170,REJECT_GUARD
10,Person,Jon Smith,Richard Gonzalez,0.517227,REJECT_GUARD
11,Company,Meeno,UJET,0.507369,REJECT_GUARD
12,Person,Hemesh Singh,Rajiv Mehrishi,0.501028,REJECT_GUARD


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [21]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()
print(f"Total nodes assigned: {len(community_df)}  |  Total communities: {community_df.community_id.nunique()}")

sizes = community_df.groupby("community_id").size().sort_values(ascending=False)
display(sizes.head(10).rename("size").to_frame())

print()
print("=== Community Reports (top 3 largest) ===")
for cid in sizes.head(3).index:
    ids = community_df[community_df.community_id == cid]["id"].tolist()
    names = pd.DataFrame(run_cypher(
        "MATCH (n:Entity) WHERE n.id IN $ids RETURN n.name AS name, n.entity_type AS type",
        ids=ids
    ))
    print()
    print(f"--- Community {cid} ({len(ids)} nodes) ---")
    print(", ".join(f"{r.name} [{r.type}]" for r in names.itertuples()))


Total nodes assigned: 103  |  Total communities: 41


,size
community_id,
0,6
1,6
2,4
3,4
4,3
5,3
6,3
7,3
8,3



=== Community Reports (top 3 largest) ===

--- Community 0 (6 nodes) ---
Ouster [Company], Bachleitner Technology [Company], Railergy [Company], ParkVi [Company], Systerel SAS [Company], Applanix [Company]

--- Community 1 (6 nodes) ---
Logic Pro [Technology], Apple [Company], M3 [Technology], A17 Bionic [Technology], Arm [Company], Final Cut Pro [Technology]



--- Community 2 (4 nodes) ---
Hemesh Singh [Person], Unacademy [Company], Gaurav Munjal [Person], Roman Saini [Person]


In [22]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# Demo: self-correction trên câu hỏi G03 (case Aaritya Technologies bị "vô hình"
# trong retrieval mặc định - xem reports/failure_analysis.md).
demo_question = "Compare two startup funding events reported in September 2023: Meeno and Aaritya Technologies. Who invested, and how do the deals differ?"
demo_result = self_correcting_context(demo_question)
print("Route lua chon:", demo_result["route"])
print("Missing (theo LLM):", demo_result["missing"])
print()
print(demo_result["context"][:2500])

# Demo: self-correction tren cau hoi G03 (case Aaritya Technologies bi
# "vo hinh" trong retrieval mac dinh - xem reports/failure_analysis.md).
demo_question = "Compare two startup funding events reported in September 2023: Meeno and Aaritya Technologies. Who invested, and how do the deals differ?"
demo_result = self_correcting_context(demo_question)
print("Route lua chon:", demo_result["route"])
print("Missing (theo LLM):", demo_result["missing"])
print()
print(demo_result["context"][:2500])


Route lua chon: hop3+vector
Missing (theo LLM): Information about Aaritya Technologies' September 2023 funding event, including the investors and deal details, is missing from the context.

=== GRAPH ===
Renate Nyborg [Person] -FOUNDED-> Meeno [Company] | date=2023-09-27 | chunk=212476770ac78703d3b7::c0000 | evidence=Renate Nyborg's startup Meeno
Sequoia [Company] -INVESTED_IN-> Meeno [Company] | date=2023-09-27 | chunk=212476770ac78703d3b7::c0000 | evidence=Meeno has raised a $3.9 million seed round led by Sequoia.

=== VECTOR ===
[chunk_id=5196c9020eb3910043f6::c0000 | date=2023-09-14 | score=0.516]
In the largest biotech Series C financing so far this year Generate:Biomedicines picked up new investors including Amgen and NVIDIA’s venture capital arm to advance its pipeline of 17 programs.

[chunk_id=212476770ac78703d3b7::c0000 | date=2023-09-27 | score=0.513]
Former Tinder CEO Renate Nyborg''s startup Meeno which aims to fight loneliness through an AI-powered chatbot announced that 

Route lua chon: hop3+vector
Missing (theo LLM): Information about Aaritya Technologies' September 2023 funding event, including the investors and deal details, is missing from the context.

=== GRAPH ===
Renate Nyborg [Person] -FOUNDED-> Meeno [Company] | date=2023-09-27 | chunk=212476770ac78703d3b7::c0000 | evidence=Renate Nyborg's startup Meeno
Sequoia [Company] -INVESTED_IN-> Meeno [Company] | date=2023-09-27 | chunk=212476770ac78703d3b7::c0000 | evidence=Meeno has raised a $3.9 million seed round led by Sequoia.

=== VECTOR ===
[chunk_id=5196c9020eb3910043f6::c0000 | date=2023-09-14 | score=0.516]
In the largest biotech Series C financing so far this year Generate:Biomedicines picked up new investors including Amgen and NVIDIA’s venture capital arm to advance its pipeline of 17 programs.

[chunk_id=212476770ac78703d3b7::c0000 | date=2023-09-27 | score=0.513]
Former Tinder CEO Renate Nyborg''s startup Meeno which aims to fight loneliness through an AI-powered chatbot announced that 

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau